In [ ]:
# @title 1. Setup & Generate requirements.txt
import sys

# 1. Define strict versions for critical libraries
# These versions are compatible with Python 3.10+ and Google Colab as of late 2024/2025
requirements = """
torch>=2.0.0
transformers==4.35.2
datasets==2.14.6
scikit-learn==1.3.2
pandas==2.1.3
numpy==1.26.2
beautifulsoup4==4.12.2
streamlit==1.28.2
accelerate==0.24.1
fastapi==0.104.1
uvicorn==0.24.0
"""

# 2. Write this to a file
with open('requirements.txt', 'w') as f:
    f.write(requirements.strip())

print("✅ requirements.txt created.")

# 3. Install strictly from this file
!pip install -r requirements.txt -q

import torch
print(f"✅ Setup Complete. Torch version: {torch.__version__}")
print("⬇️ ACTION: Download 'requirements.txt' from the Files tab on the left.")

✅ requirements.txt created.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 140.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.0/143.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.4/261.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.9/92.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [ ]:
!pip install transformers[torch] accelerate -U -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 116.5 MB/s eta 0:00:00


In [1]:
import os
import re
import torch
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, classification_report, accuracy_score
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    EvalPrediction
)
from torch.utils.data import Dataset

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
# @title 2. Data Preparation & Cleaning

# --- HELPER: HTML Cleaner ---
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove HTML tags
    soup = BeautifulSoup(text, "html.parser")
    text = soup.get_text(separator=" ")
    # Remove special chars and lower case
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

# --- SYNTHETIC DATA GENERATION (Replace this with your pd.read_csv) ---
data = pd.read_csv("products.csv", encoding="latin1")
df_raw = pd.DataFrame(data)

# --- PREPROCESSING STEPS ---

print(f"Raw record count: {len(df_raw)}")

# 1. Clean Text
df_raw['clean_desc'] = df_raw['product_description'].apply(clean_text)
df_raw['clean_name'] = df_raw['product_name'].apply(clean_text)
df_raw['combined_text'] = df_raw['clean_name'] + " " + df_raw['clean_desc']

# 2. Create Combined Labels (Parent > Subcategory)
# This creates a unique label for every parent-sub combination
df_raw['label_str'] = df_raw['parent_category'] + " > " + df_raw['sub_category']

# 3. Group by Product to handle Multi-Labels
# We group by the text content to merge labels for the same product
df_grouped = df_raw.groupby(['combined_text', 'product_name', 'product_description'])['label_str'].apply(list).reset_index()

# 4. Remove Duplicates (Logic: If text is identical, we already grouped them.
# If different text but same "product", we keep unique text entries)
df_grouped = df_grouped.drop_duplicates(subset=['combined_text'])

print(f"Unique product count after cleaning: {len(df_grouped)}")
print("Sample Record:")
print(df_grouped.head(1))

# 5. Multi-Label Binarization
mlb = MultiLabelBinarizer()
labels_matrix = mlb.fit_transform(df_grouped['label_str'])

# Save classes for later inference
id2label = {i: label for i, label in enumerate(mlb.classes_)}
label2id = {label: i for i, label in enumerate(mlb.classes_)}
num_labels = len(mlb.classes_)

print(f"\nNumber of unique category combinations: {num_labels}")

Raw record count: 6586


/tmp/ipython-input-269084699.py:8: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")
/tmp/ipython-input-269084699.py:8: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")


Unique product count after cleaning: 5541
Sample Record:
                                       combined_text  \
0  050 oz 15 ml travel strap square anitbacterial...   

                                        product_name  \
0  0.50 oz/ 15 ml Travel Strap Square Anitbacteri...   

                                 product_description            label_str  
0  Fight germs at home or on the go with our squa...  [Accessories > PPE]  

Number of unique category combinations: 68


In [3]:
# @title 3. Tokenizer & Dataset Class

# Load Tokenizer
model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)

class ProductDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)
        }

# Split Data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_grouped['combined_text'].values,
    labels_matrix,
    test_size=0.15,
    random_state=42
)

train_dataset = ProductDataset(train_texts, train_labels, tokenizer)
val_dataset = ProductDataset(val_texts, val_labels, tokenizer)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Training samples: 4709
Validation samples: 832


In [6]:

# @title 4. Model Training with Early Stopping

# Initialize Model
model = RobertaForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)
model.to(device)

# Metrics Function
# @title Updated Metrics Function (Fixes Zero F1)

def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions

    # 1. Apply Sigmoid to get probabilities (0.0 to 1.0)
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.tensor(preds))

    # --- FIX: LOWER THRESHOLD ---
    # Lowering to 0.3 catches "weak" predictions early in training.
    # As the model gets better, confidence will rise naturally.
    threshold = 0.3
    y_pred = np.zeros(probs.shape)
    y_pred[probs >= threshold] = 1

    y_true = p.label_ids

    # --- DEBUGGING PRINT ---
    # This prints the max probability of the first item in the batch
    # So you can see if the model is predicting 0.01 (broken) or 0.45 (just shy)
    print(f" [Debug] Max Prob in batch: {probs.max().item():.4f} | Threshold: {threshold}")

    # 2. Calculate Metrics with zero_division=0 to silence warnings
    f1_micro = f1_score(y_true=y_true, y_pred=y_pred, average='micro', zero_division=0)
    f1_macro = f1_score(y_true=y_true, y_pred=y_pred, average='macro', zero_division=0)
    accuracy = accuracy_score(y_true, y_pred) # Exact match accuracy

    return {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'accuracy': accuracy
    }

# Re-initialize Trainer with new metrics
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Resuming training with fixed metrics...")
trainer.train()

# Save final model
model_save_path = "./saved_model"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model saved to {model_save_path}")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Resuming training with fixed metrics...


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy
1,0.081200,0.082016,0.000000,0.000000,0.000000
2,0.077800,0.072469,0.030270,0.012478,0.014423
3,0.048900,0.048033,0.495019,0.330117,0.341346
4,0.033800,0.034906,0.695545,0.572166,0.586538
5,0.027600,0.028756,0.744881,0.662179,0.656250
6,0.022500,0.025811,0.779470,0.697637,0.717548
7,0.018000,0.022903,0.810056,0.747940,0.748798
8,0.015900,0.021639,0.827969,0.771116,0.764423
9,0.011300,0.020963,0.833056,0.778249,0.781250
10,0.012100,0.020772,0.830480,0.778091,0.771635


 [Debug] Max Prob in batch: 0.0307 | Threshold: 0.3
 [Debug] Max Prob in batch: 0.3373 | Threshold: 0.3
 [Debug] Max Prob in batch: 0.7288 | Threshold: 0.3
 [Debug] Max Prob in batch: 0.8551 | Threshold: 0.3
 [Debug] Max Prob in batch: 0.9118 | Threshold: 0.3
 [Debug] Max Prob in batch: 0.9502 | Threshold: 0.3
 [Debug] Max Prob in batch: 0.9532 | Threshold: 0.3
 [Debug] Max Prob in batch: 0.9603 | Threshold: 0.3
 [Debug] Max Prob in batch: 0.9547 | Threshold: 0.3
 [Debug] Max Prob in batch: 0.9634 | Threshold: 0.3
Model saved to ./saved_model


In [7]:
# @title 5. Evaluation & Stress Testing

# --- 1. Standard Metrics ---
print("### Classification Report ###")
predictions = trainer.predict(val_dataset)
sigmoid = torch.nn.Sigmoid()
probs = sigmoid(torch.tensor(predictions.predictions))
y_pred = np.zeros(probs.shape)
y_pred[probs >= 0.5] = 1

print(classification_report(val_labels, y_pred, target_names=mlb.classes_, zero_division=0))

# --- 2. Stress Testing Function ---
def stress_test_model(model, tokenizer, text_input):
    """Checks if model handles edge cases without crashing"""
    model.eval()
    inputs = tokenizer(text_input, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probs = torch.sigmoid(logits).cpu().numpy()[0]

    # Get predictions > 0.3 (lower threshold for stress test visibility)
    predicted_indices = np.where(probs > 0.3)[0]
    results = [(model.config.id2label[i], float(probs[i])) for i in predicted_indices]
    return results

print("\n### Stress Testing ###")
edge_cases = [
    "",                                      # Empty string
    "   ",                                   # Whitespace
    "<div></div>" * 50,                      # Repetitive HTML garbage
    "xjhsdf kjsd fksjdf",                    # Nonsense noise
    "Super " * 200                           # Very long input
]

for case in edge_cases:
    try:
        res = stress_test_model(model, tokenizer, case)
        print(f"Input: '{case[:30]}...' -> Status: OK. Preds: {res}")
    except Exception as e:
        print(f"Input: '{case[:30]}...' -> Status: FAILED. Error: {e}")

### Classification Report ###


 [Debug] Max Prob in batch: 0.9547 | Threshold: 0.3
                                        precision    recall  f1-score   support

                    Accessories > Auto       0.91      0.77      0.83        13
       Accessories > Blankets & Throws       0.92      0.92      0.92        12
    Accessories > Food & Confectionery       0.88      1.00      0.93        14
                    Accessories > Golf       0.89      0.89      0.89         9
       Accessories > Health & Wellness       0.70      0.78      0.74         9
                 Accessories > Kitchen       1.00      0.92      0.96        12
                  Accessories > Others       0.79      0.68      0.73        22
                Accessories > Outdoors       0.75      0.50      0.60        18
                     Accessories > PPE       1.00      0.79      0.88        19
                 Accessories > Patches       1.00      0.90      0.95        10
                    Accessories > Pets       0.94      0.89      0.

In [8]:
!zip -r saved_model.zip ./saved_model

  adding: saved_model/ (stored 0%)
  adding: saved_model/model.safetensors (deflated 13%)
  adding: saved_model/training_args.bin (deflated 53%)
  adding: saved_model/merges.txt (deflated 53%)
  adding: saved_model/config.json (deflated 72%)
  adding: saved_model/vocab.json (deflated 68%)
  adding: saved_model/tokenizer_config.json (deflated 76%)
  adding: saved_model/special_tokens_map.json (deflated 84%)
